# Context-Aware Text-to-3D Scene Generation
### End-to-End Google Colab Walkthrough

This notebook demonstrates the 6-stage cascaded pipeline:
1. **Text-to-Image Generation** (FLUX Diffusion Model)
2. **Object Segmentation** (Segment Anything Model - SAM)
3. **3D Asset Reconstruction** (2D-to-3D Depth Lifting)
4. **6D Pose Translation & Scene Graph** (Collision-aware positioning)
5. **Material Generation** (SVBRDF PBR Maps)
6. **Scene Composition & Visualization**

In [ ]:
# Step 0: Environment Setup & Dependencies
!git clone https://github.com/facebookresearch/segment-anything.git
!pip install -q diffusers transformers accelerate torch torchvision open3d trimesh opencv-python matplotlib
!mkdir -p models outputs/masks outputs/assets outputs/materials outputs/scenes

In [ ]:
# Download SAM Checkpoint (ViT-B or ViT-H)
!wget -nc https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth -P models/

## 1. Text-to-Image Generation (FLUX)
Generate an isometric 2D scene layout guided by a natural language prompt.

In [ ]:
import torch
from diffusers import FluxPipeline
from PIL import Image

prompt = (
    "Isometric view of a modern aesthetic bathroom with pastel teal walls, "
    "ceramic bathtub, washbasin, mirror, and commode, soft studio lighting"
)

pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()

scene_image = pipe(prompt, num_inference_steps=4, guidance_scale=0.0).images[0]
scene_image.save("outputs/01_scene_layout.png")
scene_image

## 2. Object Segmentation (SAM)
Isolate individual scene objects into masks and transparent RGBA cutouts.

In [ ]:
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
import cv2
import numpy as np
import matplotlib.pyplot as plt

sam = sam_model_registry["vit_b"](checkpoint="models/sam_vit_b_01ec64.pth").to("cuda")
mask_gen = SamAutomaticMaskGenerator(sam, min_mask_region_area=2000)

img_bgr = cv2.imread("outputs/01_scene_layout.png")
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
masks = mask_gen.generate(img_rgb)

print(f"Detected {len(masks)} candidate objects")

# Display top 4 masks
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, ax in enumerate(axes):
    if i < len(masks):
        ax.imshow(masks[i]['segmentation'], cmap='gray')
        ax.set_title(f"Object {i+1}")
        ax.axis('off')
plt.show()

## 3. 3D Asset Reconstruction (2D-to-3D Lifting)
Estimate depth and build watertight polygon meshes with Trimesh.

In [ ]:
import trimesh

# Generate simple watertight 3D mesh from first segmented object
mask_0 = masks[0]['segmentation']
y_coords, x_coords = np.where(mask_0)
pts_front = np.stack([x_coords / 512.0, -y_coords / 512.0, np.ones_like(x_coords) * 0.15], axis=1)
pts_back = np.stack([x_coords / 512.0, -y_coords / 512.0, -np.ones_like(x_coords) * 0.15], axis=1)
pts = np.vstack([pts_front, pts_back])

mesh = trimesh.convex.convex_hull(pts)
mesh.export("outputs/assets/object_00.obj")
print("Exported 3D mesh:", mesh)

## 4. Pose Translation & Scene Graph
Align objects in a shared metric room coordinate system and export `scene_graph.json`.

In [ ]:
import json

scene_graph = {
    "scene_name": "Colab_Isometric_Bathroom",
    "room_dimensions": {"width": 3.0, "depth": 3.0, "height": 2.6},
    "objects": [
        {
            "name": "bathtub",
            "mesh": "outputs/assets/object_00.obj",
            "position": [-0.8, 0.35, -0.7],
            "rotation": [0, 45, 0],
            "scale": [1.2, 0.7, 0.8]
        }
    ]
}

with open("outputs/scene_graph.json", "w") as f:
    json.dump(scene_graph, f, indent=2)

print(json.dumps(scene_graph, indent=2))